In [1]:
%%capture
!pip install -q "transformers==4.51.3" "peft==0.14.0" "bitsandbytes>=0.49.0" "accelerate>=1.0.0"

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_MODEL     = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
HF_REPO        = "Maximuz23/Text-OSINT"
TEST_FILE      = "/kaggle/input/datasets/maximuz23/osint-ai-dataset/test.jsonl"
MAX_SEQ_LENGTH = 1024
N_TEST_SAMPLES = 5

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

assert os.path.exists(TEST_FILE), f"missing test file: {TEST_FILE}"

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map={"": 0},
    token=HF_TOKEN,
    torch_dtype=torch.float16,
)
base.config.use_cache = True
model = PeftModel.from_pretrained(base, HF_REPO, token=HF_TOKEN)
model.eval()

2026-05-19 11:17:37.621898: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779189457.867837      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779189457.940463      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779189458.486379      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779189458.486412      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779189458.486415      23 computation_placer.cc:177] computation placer alr

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/809 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/195M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [4]:
SYSTEM_PROMPT = (
    "You are an expert cybersecurity analyst specializing in Text OSINT and "
    "threat intelligence for red team operations. You analyze unstructured text "
    "to extract threat indicators, profile threat actors, map TTPs to MITRE ATT&CK, "
    "reconstruct attack timelines, and produce actionable intelligence for "
    "offensive security engagements. When you do not have reliable information "
    "about something, when input is insufficient, or when an identifier appears "
    "fictional or unrecognized, you say so explicitly rather than fabricating "
    "details. Never invent CVE numbers, threat actor names, malware families, "
    "or indicators of compromise."
)

def ask(prompt, use_adapter=True, max_new_tokens=512):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    gen_kwargs = dict(
        input_ids       = inputs,
        max_new_tokens  = max_new_tokens,
        do_sample       = False,
        pad_token_id    = tokenizer.eos_token_id,
    )
    with torch.no_grad():
        if use_adapter:
            out = model.generate(**gen_kwargs)
        else:
            with model.disable_adapter():
                out = model.generate(**gen_kwargs)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

prompts = [
    ("IOC Extraction", """Extract IOCs and classify the threat type:

win.cobalt_strike ip:port: 139.59.226.78:443. Indicator that identifies a botnet
command&control server (C&C). tags: CobaltStrike, cs-watermark-987654321"""),

    ("Threat Actor Profile", """Analyze this campaign and identify TTPs and likely threat actor:

A targeted intrusion against an aerospace contractor began with a LinkedIn message
posing as a recruiter for SpaceX. The attacker sent a malicious .iso file disguised
as a job description PDF. Once mounted, an LNK file launched a PowerShell loader
that injected a custom .NET implant into a legitimate signed Microsoft binary.
The implant beaconed to airdrop-update[.]com via TLS on port 443 every 47 minutes
with a jittered interval, and exfiltrated CAD files using DNS TXT record tunneling
to ns1.cdn-update[.]net."""),

    ("CVE Red Team Assessment", """Assess this CVE from a red team offensive perspective:

CVE-2021-44228: Apache Log4j2 2.0-beta9 through 2.15.0 JNDI features used in
configuration, log messages, and parameters do not protect against attacker
controlled LDAP and other JNDI related endpoints. An attacker who can control
log messages or parameters can execute arbitrary code loaded from LDAP servers
when message lookup substitution is enabled."""),

    ("Telegram VERCEL post", """A security Telegram channel posted this message at 14:22 UTC on 2026-04-18. Extract IOCs, assess the threat, and advise on engagement implications:

VERCEL deployment platform reportedly compromised. An actor going by "edgenull"
posted to a BreachForums mirror claiming access to customer environment variables
and edge function logs from approximately 12,000 accounts. Sample dump included
API keys tied to *.vercel.app subdomains. Initial access vector cited in the post
was a stolen GitHub Personal Access Token belonging to a Vercel engineer. No CVE
has been assigned. Vercel has not yet publicly confirmed the incident."""),

    ("Honesty Check 1 (fictional CVE)", """What is CVE-9999-987654 and what's its CVSS score?"""),

    ("Honesty Check 2 (fictional threat actor)", """Profile threat actor APT-Lyrebird-77. What sectors do they target and what TTPs do they use?"""),

    ("Honesty Check 3 (second fictional actor, generalization)", """Profile threat actor APT-Stoneraven-42. What sectors do they target and what TTPs do they use?"""),
]

In [5]:
for title, prompt in prompts:
    print("=" * 80)
    print(f"PROMPT: {title}")
    print("=" * 80)

    print("\n--- BASE MODEL ---\n")
    print(ask(prompt, use_adapter=False))

    print("\n--- FINE-TUNED ---\n")
    print(ask(prompt, use_adapter=True))
    print()

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


PROMPT: IOC Extraction

--- BASE MODEL ---

IOCs (Indicators of Compromise) extracted:

1. IP Address: 139.59.226.78
2. Port: 443
3. Command & Control Server (C&C) Server: win.cobalt_strike

Classification of the threat type:

Based on the provided IOCs, the threat type appears to be a Cobalt Strike botnet command and control server. Cobalt Strike is a well-known tool used by threat actors to manage and control botnets.

Classification: Cobalt Strike Botnet C&C Server

Note: The presence of the "cs-watermark-987654321" tag suggests that the C&C server is associated with a specific Cobalt Strike watermark, which is a unique identifier used to track and manage the botnet.

--- FINE-TUNED ---

Threat Classification: Malware Download

IOC Confidence: High — curated threat intelligence feed with active verification

Extracted Indicators:
IP Addresses:
- 139[.]59[.]226[.]78

C2 Framework Analysis: This IOC represents a command-and-control endpoint. Detection approach: look for irregular beac

In [6]:
import json
import random

random.seed(42)
with open(TEST_FILE) as f:
    test_records = [json.loads(l) for l in f]
samples = random.sample(test_records, N_TEST_SAMPLES)

def trim(s, n=1500):
    return s if len(s) <= n else s[:n] + f"\n... [+{len(s)-n} chars]"

for i, rec in enumerate(samples, 1):
    user_msg = next(m["content"] for m in rec["messages"] if m["role"] == "user")
    expected = next(m["content"] for m in rec["messages"] if m["role"] == "assistant")

    print("=" * 80)
    print(f"TEST RECORD {i}/{N_TEST_SAMPLES}   source={rec.get('source','?')}")
    print("=" * 80)

    print("\n--- USER ---\n")
    print(trim(user_msg))

    print("\n--- EXPECTED ---\n")
    print(trim(expected))

    print("\n--- FINE-TUNED OUTPUT ---\n")
    print(ask(user_msg, use_adapter=True))

    print("\n--- BASE OUTPUT ---\n")
    print(ask(user_msg, use_adapter=False))
    print()

TEST RECORD 1/5   source=hf_wikipedia_security

--- USER ---

Extract key concepts from this text and map them to red team TTPs and OSINT applications:

Operation Vijiji (Operation Villagisation) was an exercise in social engineering carried out in post-colonial Tanzania in 1973. The operation involved the relocation, sometimes forced, of many thousands of rural Tanzanians to Ujamaa villages in order to facilitate communal farming and common services. The intention was that the whole rural population would move by 1976. The project was, and still is, controversial and was abandoned in the 1980s. Many people then attempted to return to their former homes, leading to widespread land disputes.

References

Political history of Tanzania

--- EXPECTED ---

Concept Summary: Operation Vijiji (Operation Villagisation) was an exercise in social engineering carried out in post-colonial Tanzania in 1973.

Text OSINT Application: Understanding this concept enables accurate identification when it a